# Train one kind

Fine-tunes one decision kind from the Laya base on its labelled splits, calibrates it, scores validation, and runs the recursive improvement loop. Needs the `laya` package, the base checkpoint and a GPU, all installed only with approval. Labelling, note writing and generation are paid host turns: set a turn limit and get approval before running those cells.

Expects `laya/data/<kind>/{train,calibration,validation}.jsonl` from `laya/label.py` (plan stages 2 and 3). The gate test is never read here.

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
LAYA = HERE.parent if HERE.name == "notebooks" else HERE / "laya"
sys.path.insert(0, str(LAYA))   # this folder's modules by bare name; `import laya` stays the package

import label
import loop
import notes
import train

KIND = "useful"   # the pilot kind
for split in ("train", "calibration", "validation"):
    rows = label.read(label.DATA / KIND / f"{split}.jsonl")
    print(split, len(rows), "states", sum(len(r["expected"]) for r in rows), "questions")

## Baseline

Train on `train` alone and score validation.

In [ ]:
checkpoint = train.train(KIND, ["train"])
wrong, predicted = train.score(KIND, checkpoint)
universe = loop.items(label.read(label.DATA / KIND / "validation.jsonl"))
print(f"validation accuracy {1 - len(wrong) / len(universe):.3f} on {len(universe)} questions")
for key, errors, size in loop.slices(label.read(label.DATA / KIND / "validation.jsonl"), wrong)[:10]:
    print(key, errors, "/", size)

## The loop (paid)

`generate(kind, slice, n, brief, round)` writes `aug-<round>.candidates.jsonl` through the production request builders (plan stage 2) and is not built yet; the loop cannot run without it. `LABELLERS`, `MAX_TURNS` and `WRITER` are the approved run's values.

In [ ]:
LABELLERS = ["opus", "codex:gpt-6.1-sol"]
MAX_TURNS = 0    # set from the approval; 0 sends nothing
WRITER = None    # label.writer("opus") to diagnose notebook pages (paid)


def generate(kind, slice, n, brief, round):
    raise NotImplementedError("plan stage 2: build candidates through the production request builders")


def label_round(kind, split, target, round, slice):
    return label.label(kind, split, target, LABELLERS, MAX_TURNS, round=round, slice=slice)


result = loop.run(KIND, train=train.train, score=train.score, generate=generate, label=label_round, writer=WRITER)
result

## The error notebook

Every page with its status; `notes.brief` is what the next round's generator reads.

In [ ]:
print((notes.NOTES / "index.md").read_text(encoding="utf-8") if (notes.NOTES / "index.md").exists() else "no pages yet")
notes.brief(KIND)["avoid"]